# Gemini API のトークン数と使用量を確かめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/gemini_count_tokens.ipynb)

NLP クックブック 13-16 ／ 大規模言語モデル (LLM)

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

送る前にトークン数を数えて、コンテキスト長や無料枠の上限に収まるかを確かめたい。日本語と英語でトークン数がどれくらい違うかも知りたい。

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"

samples = {
    "日本語": "吾輩は猫である。名前はまだ無い。どこで生れたかとんと見当がつかぬ。",
    "英語": "I am a cat. As yet I have no name. I have no idea where I was born.",
}
for lang, s in samples.items():
    n = client.models.count_tokens(model=MODEL, contents=s).total_tokens
    print(f"{lang}: {len(s)} 文字 → {n} トークン ({len(s) / n:.2f} 文字/トークン)")

In [ ]:
# 実際に生成したときの使用量 (入力・出力・思考) を記録する
response = client.models.generate_content(model=MODEL, contents="上の 2 つの文の違いを 1 文で説明してください。\n" + "\n".join(samples.values()))
u = response.usage_metadata
print({"入力": u.prompt_token_count, "出力": u.candidates_token_count, "思考": u.thoughts_token_count, "合計": u.total_token_count})

# 大量に処理する前に、数件で 1 件あたりのトークン数を測り、全体の量と処理時間を見積もる
n_docs, per_doc = 10_000, u.total_token_count
print(f"{n_docs} 件なら約 {n_docs * per_doc / 1e6:.1f}M トークン")

- トークン数はモデルのトークナイザで決まる。他社のトークナイザ (tiktoken など) での推定は当てにならないので、使うモデルで数える。
- 無料枠では料金はかからないが、1 分あたりのトークン数やリクエスト数の上限がある。大量の文書を処理する前に、件数 × 1 件あたりのトークン数で、上限に収まるかを見積もる。
- 思考 (thinking) をするモデルでは、出力とは別に思考のトークンも消費する。単純な処理なら思考を止める (`thinking_budget=0`)。
- トークン数のカウント自体は無料で、生成のリクエスト数にも数えられない (レート制限はある)。